# 3장 실습 — Keras의 백엔드를 갈아 끼운다

Keras 3는 TensorFlow · PyTorch · JAX **위에서 도는 상위 API**입니다.
엔진을 바꿔도 코드는 한 글자도 안 고칩니다.

> ⚠ 아래 첫 셀의 환경변수는 **`import keras` 보다 위**에 있어야 합니다.
> 이미 keras가 불려 온 뒤에 바꾸면 적용되지 않습니다.
> 백엔드를 바꿔 다시 돌리시려면 **커널을 재시작하고 처음부터** 실행하십시오.

## 3.4.1 백엔드를 고릅니다

`"tensorflow"` 또는 `"torch"` 로 바꿔 가며 실행해 보십시오.

In [1]:
import os

# ← 여기를 "tensorflow" 또는 "torch" 로 바꿉니다
os.environ["KERAS_BACKEND"] = "tensorflow"

try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

## 3.4.2 같은 코드를 돌립니다

아래는 §3.2~3.3의 Keras 판과 **완전히 같은 코드**입니다.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print("백엔드:", keras.backend.backend())
print(dlbook.versions())

x, y = data.apples(n=400, seed=42)
s = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)

model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1),
              loss="binary_crossentropy", metrics=["accuracy"])
model.fit(s.x_train, s.y_train, validation_data=(s.x_val, s.y_val),
          epochs=dlbook.smoke.epochs(60), batch_size=16, verbose=0)

pred = (model.predict(s.x_test, verbose=0).reshape(-1) > 0.5).astype(int)
dlbook.record(f"ch03_backend_{keras.backend.backend()}_acc",
              metrics.accuracy(s.y_test, pred))

백엔드: tensorflow
{'python': '3.12.3', 'numpy': '2.1.3', 'keras': '3.15.1', 'tensorflow': '2.21.0', 'torch': '-', 'keras_backend': 'tensorflow'}


ch03_backend_tensorflow_acc = 0.9500


0.95

## 정리

같은 코드가 **다른 엔진 위에서 같은 답**을 냈습니다.

그러니 §3.3에서 본 세 판의 차이는 **딥러닝의 차이가 아니라 API의 차이**입니다.

이 실험은 저장소의 CI가 매번 자동으로 돌립니다.
백엔드를 바꿔 결과가 달라지면 빌드가 실패합니다.